[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C33_Context_Memory_Course/05_prompt_caching/05_prompt_caching.ipynb)

# 05 · Prompt 缓存与长会话（prompt caching）

目标：用**纯标准库**从零模拟 prompt 缓存——**前缀哈希 → 命中/未命中 → usage 三字段 → TTL 过期 → 成本账 → 盈亏平衡**，全程 `assert` 验证、**无需 API key**。

路线：近似计数 → 前缀渲染 → PromptCache(命中/写入) → usage 三字段 → 成本计算 → TTL(逻辑时钟) → ✏️ 练习 → 📖 答案 → 🧪 长会话成本账胶囊。

> 心智模型：**缓存 = 把每轮都重发的稳定前缀「记住」，下次照搬不重算**。省钱前提是前缀逐字节稳定；难点不在「开启缓存」，而在保证前缀可复用并算清它的账。

## 1 · 近似 token 计数（与全课一致）

成本按 token 算，先要会数 token。沿用全课的**确定性近似**计数（真实请改用 `client.messages.count_tokens`，**绝不要用别家 tokenizer**）。

In [ ]:
import hashlib, json

def count_tokens(text):
    '''确定性近似: 按空白切词 + 长词折算, 粗估 token 数。'''
    if not text:
        return 0
    return sum(max(1, (len(w) + 3) // 4) for w in text.split())

assert count_tokens('') == 0
assert count_tokens('hello world') >= 2
# 单调: 更长文本 token 不更少
assert count_tokens('a b c d e') >= count_tokens('a b c')
print('近似 token (一段系统提示):', count_tokens('You are a helpful coding assistant. ' * 20))
print('✅ 计数就位: 可重复、单调；真实换成 count_tokens 即可')

## 2 · 渲染前缀并哈希成缓存键

Anthropic 的渲染顺序是 **`tools → system → messages`**；缓存看的是「从开头到断点为止的**精确字节**」。

我们把「断点之前的稳定部分」渲染成一个字符串，再哈希成缓存键。**一个字节变 → 哈希变 → 缓存失效**——这就是前缀匹配的全部。

In [ ]:
def render_prefix(tools, system):
    '''把稳定前缀(工具+系统提示)按固定顺序、确定序列化渲染成字节串。
       注意 sort_keys=True: 防止 dict 顺序变化导致字节变化(静默失效源之一)。'''
    parts = []
    parts.append('TOOLS:' + json.dumps(tools, ensure_ascii=False, sort_keys=True))
    parts.append('SYSTEM:' + system)
    return '\n'.join(parts)

def cache_key(prefix_text):
    '''缓存键 = 前缀精确字节的哈希。'''
    return hashlib.md5(prefix_text.encode('utf-8')).hexdigest()

TOOLS = [{'name': 'search', 'description': '搜索知识库'},
         {'name': 'calc', 'description': '做算术'}]
SYSTEM = 'You are a helpful assistant. ' * 30   # 稳定的系统提示

pfx = render_prefix(TOOLS, SYSTEM)
k1 = cache_key(pfx)
# 同样的 tools+system -> 同样的键(可命中)
k2 = cache_key(render_prefix(TOOLS, SYSTEM))
# 系统提示动一个字 -> 键就变(失效)
k3 = cache_key(render_prefix(TOOLS, SYSTEM + '!'))
# 工具顺序反过来: 因为 sort_keys 是对 dict 的键排序, 但列表顺序仍重要 -> 键变
k4 = cache_key(render_prefix(list(reversed(TOOLS)), SYSTEM))
print('前缀 token:', count_tokens(pfx))
print('k1==k2 (完全相同)? ', k1 == k2)
print('k1==k3 (系统提示+!)? ', k1 == k3)
print('k1==k4 (工具顺序反转)?', k1 == k4)
assert k1 == k2          # 逐字节相同 -> 命中
assert k1 != k3          # 一个字节变 -> 失效
assert k1 != k4          # 工具顺序变 -> 失效
print('✅ 前缀哈希: 逐字节相同才同键; 任意改动都换键 —— 这就是前缀匹配')

## 3 · PromptCache：命中/写入 与 usage 三字段

现在写缓存本体。一次请求 = 稳定前缀 + 可变后缀。缓存按前缀键查找：

- **命中**（键已存在且未过期）→ 前缀 token 记为 `cache_read_input_tokens`（约 0.1× 价）。
- **未命中**（键不存在/已过期）→ 前缀 token 记为 `cache_creation_input_tokens`（约 1.25× 价），并写入缓存。
- 后缀那段每轮都变，永远是全价 `input_tokens`。

**恒等式：三者之和 = 完整 prompt 的 token 总数。** 先不管 TTL（下面再加），这一节聚焦命中/写入与三字段。

In [ ]:
class PromptCache:
    '''最小 prompt 缓存模拟: 按前缀键存 (前缀token数, 写入时刻)。
       process() 返回 usage-like 三字段 dict, 形如真实 response.usage。'''
    def __init__(self, ttl=300):
        self.store = {}        # key -> (prefix_tokens, written_at)
        self.ttl = ttl         # 秒; 5min=300, 1h=3600
        self.hits = 0
        self.misses = 0
    def process(self, prefix_text, suffix_text, now=0):
        '''now: 逻辑时钟(秒)。返回 usage 三字段。'''
        ptok = count_tokens(prefix_text)
        stok = count_tokens(suffix_text)
        key = cache_key(prefix_text)
        entry = self.store.get(key)
        fresh = entry is not None and (now - entry[1]) <= self.ttl
        if fresh:
            self.hits += 1
            usage = {'input_tokens': stok,
                     'cache_creation_input_tokens': 0,
                     'cache_read_input_tokens': ptok}
        else:
            self.misses += 1
            self.store[key] = (ptok, now)      # (重新)写入缓存
            usage = {'input_tokens': stok,
                     'cache_creation_input_tokens': ptok,
                     'cache_read_input_tokens': 0}
        return usage

cache = PromptCache(ttl=300)
SUFFIX = 'User: 第一个问题是什么？'         # 每轮都变的部分
u1 = cache.process(pfx, SUFFIX, now=0)        # 第一次: 写入
u2 = cache.process(pfx, 'User: 第二个问题', now=10)  # 命中(10s < 300)
print('第1次 usage:', u1)
print('第2次 usage:', u2)
ptok = count_tokens(pfx)
# 第一次: 前缀写入, 命中读为 0
assert u1['cache_creation_input_tokens'] == ptok and u1['cache_read_input_tokens'] == 0
# 第二次: 前缀命中读取, 写入为 0
assert u2['cache_read_input_tokens'] == ptok and u2['cache_creation_input_tokens'] == 0
# 恒等式: 三者之和 = 完整 prompt token
for u, suf in [(u1, SUFFIX), (u2, 'User: 第二个问题')]:
    total = u['input_tokens'] + u['cache_creation_input_tokens'] + u['cache_read_input_tokens']
    assert total == count_tokens(pfx) + count_tokens(suf)
assert cache.hits == 1 and cache.misses == 1
print('✅ 缓存命中/写入正确; usage 三字段之和 = 完整 prompt token')

## 4 · 成本：把 usage 三字段换成钱

把三字段按各自单价加总即得这次请求的（输入侧）成本。单价（相对全价）：**全价 1.0× · 写入 1.25× · 读取 0.1×**。

用 `base` 表示每 token 的全价（如 $3/1M = 3e-6）。成本 = `input×1.0 + creation×1.25 + read×0.1`，再乘 `base`。

In [ ]:
def usage_cost(usage, base=1.0, write_mult=1.25, read_mult=0.1):
    '''按三字段各自单价算输入侧成本(单位: base 个全价token)。'''
    return base * (usage['input_tokens'] * 1.0
                   + usage['cache_creation_input_tokens'] * write_mult
                   + usage['cache_read_input_tokens'] * read_mult)

ptok = count_tokens(pfx)
s1tok = count_tokens(SUFFIX)
s2tok = count_tokens('User: 第二个问题')
cost1 = usage_cost(u1)   # 写入轮
cost2 = usage_cost(u2)   # 命中轮
print(f'前缀 {ptok} tok | 写入轮成本 {cost1:.1f} | 命中轮成本 {cost2:.1f} (全价token当量)')
# 闭式核对(base=1): 写入轮 = 后缀全价 + 前缀×1.25; 命中轮 = 后缀全价 + 前缀×0.1
assert abs(cost1 - (s1tok * 1.0 + ptok * 1.25)) < 1e-9
assert abs(cost2 - (s2tok * 1.0 + ptok * 0.1)) < 1e-9
# 命中轮的前缀部分只花 0.1×, 远便宜于写入轮的 1.25×
assert (ptok * 0.1) < (ptok * 1.25)
print('✅ 成本 = 三字段×各自单价; 命中轮的前缀成本仅为写入轮的约 1/12.5')

## 5 · TTL 过期（逻辑时钟，绝不真 sleep）

缓存有 **TTL**：默认 5min(300s)、可选 1h(3600s)。**两次请求间隔超过 TTL，缓存在间隙里过期**，下次同前缀又是一次全额写入。

我们用一个**逻辑时钟 `now`（直接传秒数）**模拟时间流逝——**不调用 `time.sleep`**（那样会拖慢/卡住 notebook），纯靠传入不同的 `now` 值。

In [ ]:
cache2 = PromptCache(ttl=300)        # 5 分钟
uA = cache2.process(pfx, 'q1', now=0)       # 写入 @0s
uB = cache2.process(pfx, 'q2', now=200)     # 200s < 300 -> 命中
uC = cache2.process(pfx, 'q3', now=600)     # 600-200>300? 看的是与【写入时刻】的差: 600-0=600>300 -> 过期, 重新写入
uD = cache2.process(pfx, 'q4', now=620)     # 620-600=20 <300 -> 命中(读新写入的)
print('@0s   写入? ', uA['cache_creation_input_tokens'] > 0)
print('@200s 命中? ', uB['cache_read_input_tokens'] > 0)
print('@600s 过期重写?', uC['cache_creation_input_tokens'] > 0)
print('@620s 命中? ', uD['cache_read_input_tokens'] > 0)
assert uA['cache_creation_input_tokens'] > 0    # 首次写入
assert uB['cache_read_input_tokens'] > 0        # TTL 内命中
assert uC['cache_creation_input_tokens'] > 0    # 超 TTL -> 过期 -> 重新写入
assert uD['cache_read_input_tokens'] > 0        # 重写后再次命中
assert cache2.hits == 2 and cache2.misses == 2
print('✅ TTL: 间隔超 TTL 即在间隙过期、下次重写; 用逻辑时钟模拟, 不真 sleep')

## 6 · 盈亏平衡：缓存几次才回本

缓存写入(1.25×)比全价(1×)还贵——只用一次反而亏。要复用够多次才回本。

对同一前缀连发 `n` 次（都在 TTL 内）：缓存总成本 = `P×1.25 + (n-1)×P×0.1`；不缓存 = `n×P×1.0`。求最小的 `n` 使缓存更便宜。

In [ ]:
def prefix_cost_cached(P, n, write_mult=1.25, read_mult=0.1):
    '''同前缀连发 n 次(TTL 内)的前缀部分成本(缓存)。'''
    return P * write_mult + (n - 1) * P * read_mult

def prefix_cost_uncached(P, n):
    return n * P * 1.0

def break_even_n(P=1000, write_mult=1.25, read_mult=0.1):
    '''最小的 n 使缓存严格更便宜。'''
    n = 1
    while prefix_cost_cached(P, n, write_mult, read_mult) >= prefix_cost_uncached(P, n):
        n += 1
        if n > 100:
            break
    return n

# 5min 档(写1.25): n=1 时 1.25>1.0 亏; n=2 时 1.35<2.0 -> 平衡点=2
be_5m = break_even_n(write_mult=1.25)
# 1h 档(写2.0): n=1 亏; n=2 时 2.0+0.1=2.1>2.0 仍亏; n=3 时 2.2<3.0 -> 平衡点=3
be_1h = break_even_n(write_mult=2.0)
print('5min 档盈亏平衡: 第', be_5m, '次起更便宜')
print('1h   档盈亏平衡: 第', be_1h, '次起更便宜')
assert be_5m == 2          # 5min: 2 次即回本
assert be_1h == 3          # 1h: 需 3 次
# 用一次就亏: n=1 缓存比不缓存贵
assert prefix_cost_cached(1000, 1) > prefix_cost_uncached(1000, 1)
print('✅ 盈亏平衡: 5min≈2次回本、1h≈3次; 只用一次反而更贵 —— 缓存是先投入后省钱')

---
## ✏️ 练习 1：缓存键——抓出「未排序 JSON」这个静默失效源

`render_prefix` 里我们用了 `sort_keys=True` 防止 dict 顺序变化毁掉前缀。本练习反过来：写一个**有 bug 的**渲染器 `render_prefix_buggy`，它对 tools **不排序**（`json.dumps(tools)` 不带 `sort_keys`）。然后写 `keys_equal(toolsA, toolsB)`：判断两份「键顺序不同但内容相同」的工具定义，经 buggy 渲染后缓存键**是否仍相同**。

目标：体会「同样内容、不同 dict 顺序 → buggy 渲染下键不同 → 缓存失效」，而正确渲染(sort_keys)下键相同。

In [ ]:
def render_prefix_buggy(tools, system):
    # TODO: 故意 *不* 用 sort_keys —— 复现静默失效源
    #   parts = ['TOOLS:' + json.dumps(tools, ensure_ascii=False),   # 注意: 无 sort_keys
    #            'SYSTEM:' + system]
    #   return '\n'.join(parts)
    raise NotImplementedError

def keys_equal(toolsA, toolsB, system, renderer):
    # TODO: 用 renderer 分别渲染 (toolsA, system) 和 (toolsB, system),
    #       返回两者 cache_key 是否相同(bool)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# 同一个工具, 两种 dict 键顺序(内容完全相同)
tA = [{'name': 'search', 'description': '搜索'}]
tB = [{'description': '搜索', 'name': 'search'}]   # 键顺序不同, 内容相同
sys_p = 'You are helpful.'
# buggy 渲染(不排序): 键顺序不同 -> 缓存键不同 -> 会 miss
assert keys_equal(tA, tB, sys_p, render_prefix_buggy) is False
# 正确渲染(sort_keys): 内容相同 -> 缓存键相同 -> 能命中
assert keys_equal(tA, tB, sys_p, render_prefix) is True
print('✅ 练习 1 通过: 未排序 JSON 会让等价内容产生不同缓存键(静默失效); sort_keys 修复')

## ✏️ 练习 2：命中率

给定一串请求（每个是 `(prefix_text, suffix_text, now)`），跑过一个 `PromptCache`，统计**命中率** = 命中次数 / 总请求数。

实现 `hit_rate(cache, requests)`：依次 `cache.process(...)` 每个请求，返回命中率（float）。（提示：命中当且仅当该次 `cache_read_input_tokens > 0`。）

In [ ]:
def hit_rate(cache, requests):
    # TODO: 依次处理每个 (prefix, suffix, now); 统计 cache_read_input_tokens>0 的比例
    #       返回 命中次数 / 总次数 (float)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
ca = PromptCache(ttl=300)
# 4 个请求: 同一前缀 pfx; 第1次必 miss(写入), 后3次在 TTL 内 -> 命中 -> 命中率 3/4
reqs = [(pfx, 'q1', 0), (pfx, 'q2', 10), (pfx, 'q3', 20), (pfx, 'q4', 30)]
r = hit_rate(ca, reqs)
print('命中率:', r)
assert abs(r - 0.75) < 1e-9       # 3/4
# 全是不同前缀 -> 全 miss -> 命中率 0
cb = PromptCache(ttl=300)
reqs2 = [(pfx + str(i), 'q', i) for i in range(5)]
assert hit_rate(cb, reqs2) == 0.0
print('✅ 练习 2 通过: 命中率统计正确')

## ✏️ 练习 3：TTL 过期判定

不依赖 `PromptCache`，单独写过期逻辑。给定**写入时刻** `written_at`、**当前时刻** `now`、**TTL** `ttl`（都为秒），实现 `is_fresh(written_at, now, ttl)`：返回缓存是否仍新鲜（未过期）。

规则：新鲜 ⟺ `now - written_at <= ttl`。再实现 `would_hit(written_at, now, ttl)` = 同义包装（便于读）。

In [ ]:
def is_fresh(written_at, now, ttl):
    # TODO: 返回 (now - written_at) <= ttl
    raise NotImplementedError

def would_hit(written_at, now, ttl):
    # TODO: 直接返回 is_fresh(...)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert is_fresh(0, 200, 300) is True       # 200<=300 新鲜
assert is_fresh(0, 300, 300) is True       # 边界: 恰好等于 TTL 仍算新鲜
assert is_fresh(0, 301, 300) is False      # 超 1 秒 -> 过期
assert would_hit(100, 350, 300) is True    # 350-100=250<=300
assert would_hit(100, 500, 300) is False   # 500-100=400>300
print('✅ 练习 3 通过: TTL 过期判定正确(含边界)')

## ✏️ 练习 4：长会话成本与节省比例

把成本账写成代码。一个会话：稳定前缀 `P` 个 token，每轮新增可变部分 `V` 个 token（**简化：可变部分每轮独立、都全价**），共 `N` 轮，各轮在 TTL 内（前缀缓存不过期）。

实现 `session_cost(P, V, N, cached)`：返回该会话**输入侧总成本**（base=1）。
- `cached=False`：每轮前缀都全价 → 前缀部分 `N×P×1.0`，可变部分 `N×V×1.0`。
- `cached=True`：前缀第 1 轮写入 `P×1.25`、之后 `(N-1)×P×0.1`，可变部分仍 `N×V×1.0`。

再实现 `savings(P, V, N)` = `1 - cached/uncached`（节省比例）。

In [ ]:
def session_cost(P, V, N, cached, write_mult=1.25, read_mult=0.1):
    # TODO:
    #   variable = N * V * 1.0                      # 可变部分总是全价
    #   if cached: prefix = P*write_mult + (N-1)*P*read_mult
    #   else:      prefix = N * P * 1.0
    #   return prefix + variable
    raise NotImplementedError

def savings(P, V, N):
    # TODO: 1 - session_cost(...,cached=True)/session_cost(...,cached=False)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
P, V, N = 10000, 200, 50
unc = session_cost(P, V, N, cached=False)
cac = session_cost(P, V, N, cached=True)
print(f'不缓存 {unc:.0f} | 缓存 {cac:.0f} | 节省 {savings(P,V,N)*100:.1f}%')
# 闭式核对
assert abs(unc - (50*10000*1.0 + 50*200*1.0)) < 1e-6      # 500000 + 10000
assert abs(cac - (10000*1.25 + 49*10000*0.1 + 50*200*1.0)) < 1e-6  # 12500+49000+10000
# 大前缀长会话应省很多(>60%)
assert savings(P, V, N) > 0.6
# 只有 1 轮时缓存反而更贵 -> 节省为负
assert savings(10000, 200, 1) < 0
print('✅ 练习 4 通过: 长会话成本账 + 节省比例; 大前缀多轮省 ~80%+, 单轮反亏')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def render_prefix_buggy(tools, system):
    parts = ['TOOLS:' + json.dumps(tools, ensure_ascii=False),   # 无 sort_keys -> bug
             'SYSTEM:' + system]
    return '\n'.join(parts)

def keys_equal(toolsA, toolsB, system, renderer):
    return cache_key(renderer(toolsA, system)) == cache_key(renderer(toolsB, system))

In [ ]:
# 练习 2 参考答案
def hit_rate(cache, requests):
    if not requests:
        return 0.0
    hits = 0
    for prefix, suffix, now in requests:
        u = cache.process(prefix, suffix, now=now)
        if u['cache_read_input_tokens'] > 0:
            hits += 1
    return hits / len(requests)

In [ ]:
# 练习 3 参考答案
def is_fresh(written_at, now, ttl):
    return (now - written_at) <= ttl

def would_hit(written_at, now, ttl):
    return is_fresh(written_at, now, ttl)

In [ ]:
# 练习 4 参考答案
def session_cost(P, V, N, cached, write_mult=1.25, read_mult=0.1):
    variable = N * V * 1.0
    if cached:
        prefix = P * write_mult + (N - 1) * P * read_mult
    else:
        prefix = N * P * 1.0
    return prefix + variable

def savings(P, V, N):
    unc = session_cost(P, V, N, cached=False)
    cac = session_cost(P, V, N, cached=True)
    return 1 - cac / unc

---
## 🧪 真实数据胶囊：一个真实规模长会话的省钱账

用**真实量级**的数字跑一遍：一个客服 agent，系统提示 + 工具定义 + 一份产品 FAQ 长文档共约 **12000 token**（稳定前缀），用户每轮问题 + 模型回答约 **300 token**（可变），一次会话 **80 轮**，全程紧凑（5min TTL 内）。基础价按 Claude Sonnet 输入 **$3/1M token**。

> 形状对照：真实里这就是「`system` 放 FAQ 并打 `cache_control:{type:"ephemeral"}`、每轮把新问题追加到 `messages` 末尾」，命中体现在 `response.usage.cache_read_input_tokens`。

In [ ]:
# 真实量级参数
P_real = 12000      # 稳定前缀 token(系统提示+工具+FAQ)
V_real = 300        # 每轮可变 token
N_real = 80         # 会话轮数
BASE = 3e-6         # $3 / 1M token = 3e-6 美元/token

# 复用上面的 session_cost(以全价token当量计), 再乘 BASE 换成美元
unc_tok = session_cost(P_real, V_real, N_real, cached=False)
cac_tok = session_cost(P_real, V_real, N_real, cached=True)
print(f'不缓存: {unc_tok:,.0f} 全价token当量 = ${unc_tok*BASE:.4f}')
print(f'缓存:   {cac_tok:,.0f} 全价token当量 = ${cac_tok*BASE:.4f}')
print(f'每会话省: ${(unc_tok-cac_tok)*BASE:.4f}  ({savings(P_real,V_real,N_real)*100:.1f}%)')
print(f'若每天 1000 个这样的会话, 每天省 ~${(unc_tok-cac_tok)*BASE*1000:.0f}')
assert savings(P_real, V_real, N_real) > 0.6     # 大前缀长会话省 >60%
assert cac_tok < unc_tok
print('✅ 胶囊主体: 真实量级下 prompt 缓存把长会话输入成本压掉一个量级')

**🧪 胶囊练习**：实现 `daily_savings_usd(P, V, N, base, sessions_per_day)`：返回开缓存后**每天**省下的美元数。（真实做容量/成本规划就是这么估的。）

In [ ]:
def daily_savings_usd(P, V, N, base, sessions_per_day):
    # TODO: 单会话节省(token当量) * base * sessions_per_day
    #   单会话节省 = session_cost(...,False) - session_cost(...,True)
    raise NotImplementedError

In [ ]:
# 自测
d = daily_savings_usd(12000, 300, 80, 3e-6, 1000)
expected = (session_cost(12000,300,80,False) - session_cost(12000,300,80,True)) * 3e-6 * 1000
assert abs(d - expected) < 1e-6
print(f'每天省 ${d:.0f}')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def daily_savings_usd(P, V, N, base, sessions_per_day):
    per_session_tok = session_cost(P, V, N, cached=False) - session_cost(P, V, N, cached=True)
    return per_session_tok * base * sessions_per_day

---
## 🔧 旁注：对应的真实 Anthropic prompt caching 长什么样

本课模拟的前缀缓存，换成真实 Anthropic API 只是给稳定块打一个 `cache_control` 标记（伪代码，**本环境不跑、需 API key；无 key 时本课用 MockLLM 端到端**）：

```python
import anthropic
client = anthropic.Anthropic()                       # 读 ANTHROPIC_API_KEY
resp = client.messages.create(
    model='claude-sonnet-4-6', max_tokens=1024,
    system=[{                                        # 稳定前缀放 system, 打缓存断点
        'type': 'text',
        'text': SYSTEM_PROMPT + LONG_FAQ_DOC,        # 系统提示 + 长文档(每轮不变)
        'cache_control': {'type': 'ephemeral'},      # ← 缓存到这里为止的前缀; 'ttl':'1h' 可选
    }],
    messages=history + [{'role':'user','content': new_question}],  # 每轮变的放最后
)
# 验证缓存有没有真生效 —— 看 usage(本课模拟的就是它):
print(resp.usage.cache_creation_input_tokens)   # 这次写入缓存的(首轮>0)
print(resp.usage.cache_read_input_tokens)       # 这次命中读取的(后续>0 才是省到了)
print(resp.usage.input_tokens)                  # 未缓存全价部分
```

对应关系：本课 `PromptCache.process` ↔ 服务端缓存、`render_prefix`(sort_keys) ↔ 「稳定内容在前+确定序列化」、`usage` 三字段 ↔ `resp.usage` 三字段、`ttl` ↔ `cache_control` 的 5min/1h。**渲染顺序 `tools → system → messages` 与「稳定在前、易变在后」的纪律完全一致。**

### 🔧 静默失效源自查清单（真实排查必备）

开了缓存但 `cache_read_input_tokens` 始终是 0？逐项查这张表(本模块练习 1 抓的就是第 3 条)：

| 症状 | 静默失效源 | 修法 |
|------|-----------|------|
| 命中恒为 0 | 系统提示里有 `datetime.now()` / 时间戳 | 移到 `messages` 末尾(断点之后) |
| 命中恒为 0 | 前缀里有 `uuid4()` / 随机请求 ID | 移到断点之后, 或别放进 prompt |
| 命中恒为 0 | 工具/结构化 JSON 用 `json.dumps(d)` 未排序 | 一律 `sort_keys=True` |
| 命中忽高忽低 | 工具集随用户动态增删/重排(在最前) | 固定工具集与顺序 |
| 换了模型后全 miss | 缓存是按模型隔离的 | 同一会话别中途换模型 |

排查动作永远是同一个：**盯 `usage.cache_read_input_tokens`，对两次本应命中的请求做前缀 diff，抓那个变动的字节。**

### 小结
- prompt 缓存 = 把每轮都重发的**稳定前缀**在服务端记住, 命中只花约 0.1× 全价。
- **前缀匹配**: 缓存键 = 渲染前缀的精确字节; **一字节变, 该位置及之后全失效**。
- **usage 三字段**: `input` + `cache_creation`(写入1.25×) + `cache_read`(读0.1×) = 完整 prompt token; 看 `cache_read>0` 才是真省到。
- **TTL**: 5min(写1.25×)/1h(写2×); 间隔超 TTL 在间隙过期、下次重写。
- **静默失效源**: 时间戳/UUID/未排序JSON/变动工具集 → 命中恒为0; 修法是把易变的移到断点之后、JSON 排序。
- **排序比断点更重要**: 稳定在前、易变在后(`tools→system→messages`), 断点只在排好序的线上标记。
- **成本账**: 大前缀 + 多轮复用, 缓存能把输入成本压掉 ~80–90%; 但只用一次反而更贵(先投入后省钱)。

**全课到此完结**: 从 token 预算(01)→ compaction(02)→ 文件记忆(03)→ 检索(04)→ prompt 缓存(05), 你已亲手把「让一个 agent 在有限窗口里长时间不爆、不丢、不贵」的一整套上下文工程基础设施从零造了出来。把 MockLLM 换成 `client.messages.create(model='claude-sonnet-4-6', ...)`, 它们就能直接用在真实 agent 上。